An executed version of this notebook can be seen on
[IRSA's website](https://irsa.ipac.caltech.edu/docs/notebooks/neowise-source-table-lightcurves.html).

# Make Light Curves from NEOWISE Single-exposure Source Table

Learning Goals:

- Search the NEOWISE Single-exposure Source Table (Parquet version) for the light curves of a
  set of targets with RA/Dec coordinates.
  - Write a pyarrow dataset filter and use it to load the NEOWISE detections near each target (rough cut).
  - Match targets to detections using an astropy cone search (precise cut).
  - Parallelize this.
- Plot the light curves.

## 1. Introduction

This notebook loads light curves from the
[NEOWISE](https://irsa.ipac.caltech.edu/Missions/wise.html) Single-exposure Source Table
for a sample of about 2000 cataclysmic variables from [Downes et al. (2001)](https://doi.org/10.1086/320802).
The NEOWISE Single-exposure Source Table is a very large catalog -- 11 years and 42 terabytes in total
with 145 columns and 200 billion rows.
When searching this catalog, it is important to consider the requirements of your use case and
the format of this dataset.
This notebook applies the techniques developed in
[Strategies to Efficiently Work with NEOWISE Single-exposure Source Table in Parquet](https://irsa.ipac.caltech.edu/docs/notebooks/neowise-source-table-strategies.html).
This is a fully-worked example that demonstrates the important steps, but note that this is a
relatively small use case for the Parquet version of the dataset.

The specific strategy we employ is:

- Choose a cone search radius that determines which NEOWISE source detections to associate
  with each target.
- Load the sample of CV targets.
- Calculate the indexes of all HEALPix order k=5 pixels within the radius of each target.
  These are the dataset partitions that need to be searched.
- Parallelize over the partitions using `multiprocessing.Pool`.
  For each pixel:
  - Construct a dataset filter for NEOWISE sources in the vicinity of the targets in the partition.
  - Load data, applying the filter. In our case, the number of rows loaded will be fairly small.
  - Do a cone search to match sources with targets in the partition.
  - Return the results.
- Concatenate the cone search results, groupby target ID, and sort by time to construct the light curves.

The efficiency of this method will increase with the number of rows needed from each partition.
For example, a cone search radius of 1 arcsec will require about 10 CPUs, 65G RAM, and
50 minutes to load the data from all 11 NEOWISE years.
Increasing the radius to 10 arcsec will return about 2.5x more rows using roughly the same resources.
Increasing the target sample size can result in similar efficiency gains.
To try out this notebook with fewer resources, use a subset of NEOWISE years.
Using one year is expected to require about 5 CPUs, 20G RAM, and 10 minutes.
These estimates are based on testing in science platform environments.
Your numbers will vary based on many factors including compute power, bandwidth, and physical distance from the data.

## 2. Imports

In [1]:
# Uncomment the next line to install dependencies if needed.
# !pip install astropy astroquery hpgeom matplotlib pandas pyarrow pyvo

In [1]:
import multiprocessing  # parallelization

import astroquery.vizier  # fetch the sample of CV targets
import hpgeom  # HEALPix math
import numpy as np  # math
import pandas as pd  # manipulate tabular data
import pyarrow.compute  # construct dataset filters
import pyarrow.dataset  # load and query the NEOWISE dataset
import pyarrow.fs  # interact with the S3 bucket storing the NEOWISE catalog
import pyvo  # TAP service for the Vizier query
from astropy import units as u  # manipulate astropy quantities
from astropy.coordinates import SkyCoord  # manipulate sky coordinates
from matplotlib import pyplot as plt  # plot light curves

# copy-on-write will become the default in pandas 3.0 and is generally more performant
pd.options.mode.copy_on_write = True

## 3. Setup

### 3.1 Define variables

First, choose which NEOWISE years to include.
Real use cases are likely to require all ten years but it can be helpful to start with
fewer while exploring to make things run faster.

In [2]:
# all years => about 11 CPU, 65G RAM, and 50 minutes runtime
YEARS = [f"year{yr}" for yr in range(1, 12)] + ["addendum"]

# To try out a smaller version of the notebook,
# uncomment the next line and choose your own subset of years.
# YEARS = [10]  # one year => about 5 CPU, 20G RAM, and 10 minutes runtime

In [3]:
# sets of columns that we'll need
FLUX_COLUMNS = ["w1flux", "w2flux"]
LIGHTCURVE_COLUMNS = ["mjd"] + FLUX_COLUMNS
COLUMN_SUBSET = ["cntr", "ra", "dec"] + LIGHTCURVE_COLUMNS

# cone-search radius defining which NEOWISE sources are associated with each target object
MATCH_RADIUS = 1 * u.arcsec

### 3.2 Load NEOWISE metadata

The metadata contains column names, schema, and row-group statistics for every file in the dataset.
We'll load it as a pyarrow dataset.

In [4]:
# This catalog is so big that even the metadata is big.
# Expect this cell to take about 30 seconds per year.

# This information can be found at https://irsa.ipac.caltech.edu/cloud_access/.
bucket = "nasa-irsa-wise"
base_prefix = "wise/neowiser/catalogs/p1bs_psd/healpix_k5"
metadata_path = (
    lambda yr: f"{bucket}/{base_prefix}/{yr}/neowiser-healpix_k5-{yr}.parquet/_metadata"
)
fs = pyarrow.fs.S3FileSystem(region="us-west-2", anonymous=True)

# list of datasets, one per year
year_datasets = [
    pyarrow.dataset.parquet_dataset(metadata_path(yr), filesystem=fs, partitioning="hive")
    for yr in YEARS
]

# unified dataset, all years
neowise_ds = pyarrow.dataset.dataset(year_datasets)

## 4. Define functions to filter and load data

These functions will be used in the next section.
Defining them here in the notebook is useful for demonstration and should work seamlessly on Linux, which includes most science platforms.
Mac and Windows users should see the note at the end of the notebook.
However, note that this use case is likely too large for a laptop and may perform poorly and/or crash if attempted.

In [5]:
import sys
sys.path.append("/Users/adamboesky/Research/long_transients/Source_Analysis")
from neowise import *

## 5. Load light curves

Load the target objects' coordinates and other info.

In [6]:
targets_df = load_targets_Downes2001(radius=MATCH_RADIUS)
targets_df.head()

,uid,GCVS,RAJ2000,DEJ2000,healpix_k5
0,7049,Cet2,42.823792,5.629917,11
1,7048,Cet1,40.722708,6.796000,33
2,6887,WX Ari,41.900917,10.593806,39
3,6885,SV Ari,51.263917,19.831361,102
4,6885,SV Ari,51.263917,19.831361,108


Search the NEOWISE Source Table for all targets (positional matches) and load the light curves.
Partitions are searched in parallel.
For targets located near partition boundaries, relevant partitions will be searched
independently for the given target and the results will be concatenated.
If searching all NEOWISE years, this may take 45 minutes or more.

In [7]:
# Group targets by partition. 'load_lightcurves_one_partition' will be called once per group.
targets_groups = targets_df.groupby("healpix_k5")
# Arguments for 'init_worker'.
init_args = (neowise_ds, COLUMN_SUBSET, MATCH_RADIUS)

# Start a multiprocessing pool and load the target light curves in parallel.
# About 1900 unique pixels in targets_df, 8 workers, 48 chunksize => ~5 chunks per worker.
nworkers = 8
chunksize = 48
with multiprocessing.Pool(nworkers, initializer=init_worker, initargs=init_args) as pool:
    lightcurves = []
    for lightcurves_df in pool.imap_unordered(
        load_lightcurves_one_partition, targets_groups, chunksize=chunksize
    ):
        lightcurves.append(lightcurves_df)
neowise_lightcurves_df = pd.concat(lightcurves).sort_values("mjd").reset_index(drop=True)

Process SpawnPoolWorker-4:
Process SpawnPoolWorker-2:
Process SpawnPoolWorker-3:
Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/queues.py", line 386, in get
    with self._rlock:
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
Traceback (most recent 

KeyboardInterrupt: 

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/_dataset_parquet.pyx", line 278, in pyarrow._dataset_parquet.ParquetFileFormat.make_fragment
  File "pyarrow/_dataset.pyx", line 118, in pyarrow._dataset._make_file_source
  File "/Users/adamboesky/opt/anaconda3/envs/long_transients/lib/python3.12/site-packages/pyarrow/util.py", line 134, in _is_path_like
    def _is_path_like(path):
    
KeyboardInterrupt


In [ ]:
neowise_lightcurves_df.head()

## 6. Plot NEOWISE light curves

The light curves will have large gaps due to the observing cadence, so we'll plot each
"epoch" separately to see them better.

In [ ]:
# get the light curves of the target with the most data
target_uid = neowise_lightcurves_df.groupby("uid").mjd.count().sort_values().index[-1]
target_df = neowise_lightcurves_df.loc[neowise_lightcurves_df.uid == target_uid]

# list of indexes that separate epochs (arbitrarily at delta mjd > 30)
epoch_idxs = target_df.loc[target_df.mjd.diff() > 30].index.to_list()
epoch_idxs = epoch_idxs + [target_df.index[-1]]  # add the final index

# make the figure
ncols = 4
nrows = int(np.ceil(len(epoch_idxs) / ncols))
fig, axs = plt.subplots(nrows, ncols, sharey=True, figsize=(3 * ncols, 2.5 * nrows))
axs = axs.flatten()
idx0 = target_df.index[0]
for i, (idx1, ax) in enumerate(zip(epoch_idxs, axs)):
    epoch_df = target_df.loc[idx0 : idx1 - 1, LIGHTCURVE_COLUMNS].set_index("mjd")
    for col in FLUX_COLUMNS:
        ax.plot(epoch_df[col], ".", markersize=3, label=col)
    ax.set_title(f"epoch {i}")
    ax.xaxis.set_ticks(  # space by 10
        range(int((ax.get_xlim()[0] + 10) / 10) * 10, int(ax.get_xlim()[1]), 10)
    )
    idx0 = idx1
axs[0].legend()
fig.supxlabel("MJD")
fig.supylabel("RAW FLUX")
fig.suptitle(f"NEOWISE light curves for target CV {target_uid}")
fig.tight_layout()
plt.show(block=False)

-----

[*] Note to Mac and Windows users:

You will need to copy the functions and imports from this notebook into a separate '.py' file and then import them in order to use the multiprocessing pool for parallelization.
In addition, you may need to load `neowise_ds` separately for each child process (i.e., worker) by adding that code to the `init_worker` function instead of passing it in as an argument.
This has to do with differences in what does / does not get copied into the child processes on different platforms.

***

## About this notebook

**Author:** Troy Raen (IRSA Developer) and the IPAC Science Platform team

**Updated:** 2025-03-07

**Contact:** [the IRSA Helpdesk](https://irsa.ipac.caltech.edu/docs/help_desk.html) with questions or reporting problems.